# estimate size of data request

In [1]:
import pandas as pd

# data request
dr = pd.read_csv(
    "https://raw.githubusercontent.com/WCRP-CORDEX/data-request-table/refs/heads/main/data-request/dreq_default.csv"
)

# cmip6 downscaling plans
plans = pd.read_csv(
    "https://raw.githubusercontent.com/WCRP-CORDEX/simulation-status/refs/heads/main/CMIP6_downscaling_plans.csv"
)

#### Define CORDEX-CMIP6 simulations

In [2]:
AVG_PER_YEAR = {
    "fx": 1,
    "mon": 12,
    "day": 365.2425,
    "1hr": 24 * 365.2425,
    "6hr": 4 * 365.2425,
}

YEARS_PER_EXPERIMENT = {
    "evaluation": 40,
    "historical": 65,
    "ssp126": 85,
    "ssp245": 85,
    "ssp370": 85,
    "ssp585": 85,
}


def approx_timesteps(freq: str, years: int) -> float:
    val = AVG_PER_YEAR.get(freq)
    if val is None:
        raise ValueError(f"Unknown frequency '{freq}'. Add it to AVG_PER_YEAR.")
    return val * years

# this is timesteps per year depending on frequency
dr["timesteps_per_year"] = dr["frequency"].apply(lambda f: approx_timesteps(f, 1))

In [8]:
# add column with years depending on experiment
balanced = plans[(plans.comments.str.contains("#EURbalanced", na=False))]
balanced["years"] = balanced["driving_experiment_id"].apply(
    lambda e: YEARS_PER_EXPERIMENT.get(e, 0)
)

# this is total years for all planned simulations
total_years = balanced["years"].sum()
total_years

/tmp/ipykernel_1162920/3071173543.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  balanced["years"] = balanced["driving_experiment_id"].apply(


np.int64(5685)

## Result

In [6]:
# this is from real data estimate with compression
bytes_per_field = 698_752

def estimate_storage(group):
    steps_per_year = group["timesteps_per_year"].sum()
    steps_total = steps_per_year * total_years
    return pd.Series({
       # "variables": group["variable_id"].nunique(),
        "timesteps_all_per_year_sum": steps_per_year,
        "timesteps_total": steps_total,
        "estimated_TB": steps_total * bytes_per_field / 1e12,
    })

per_priority = dr.groupby("priority", dropna=False).apply(estimate_storage).sort_index()
per_priority

/tmp/ipykernel_1162920/3689496847.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_priority = dr.groupby("priority", dropna=False).apply(estimate_storage).sort_index()


,timesteps_all_per_year_sum,timesteps_total,estimated_TB
priority,,,
CORE,119616.2975,6.800187e+08,475.164393
TIER1,406301.9325,2.309826e+09,1613.995877
TIER2,159655.4875,9.076414e+08,634.216276
